# 梯子 (GPU) — 語数を増やして 損失の下がり方を測る

使う前に: 左の鍵アイコン(シークレット)に `HF_TOKEN` を追加し、 ノートブックからのアクセスをオンにする。 トークンは 読み取りだけでよい。 ランタイムは「ランタイムのタイプを変更」でGPUを選ぶ。

順番に実行する。 ③の速度確認の結果を貼ってもらってから、 ④の本番を回す。

In [ ]:
# ① GPU の確認
!nvidia-smi -L
import torch
print('cuda:', torch.cuda.is_available(), torch.__version__)

In [ ]:
# ② 道具とコード
!pip -q install tokenizers huggingface_hub
!rm -rf /content/ai-lab && git clone -q https://github.com/yoojpn/ai-lab /content/ai-lab
!cd /content/ai-lab && git log --oneline -1

In [ ]:
# ③ データ (HF の tok/ 約 2.7GB)
from google.colab import userdata
from huggingface_hub import snapshot_download
snapshot_download('Youichi1/ai-lab-data', repo_type='dataset', allow_patterns=['tok/*'], local_dir='/content/hfd', token=userdata.get('HF_TOKEN'))
!rm -rf /tmp/tok && mv /content/hfd/tok /tmp/tok && ls /tmp/tok | head -5 && df -h /tmp | tail -1

In [ ]:
# ④ 速度確認 (300万語, 数分)。 『歩/秒』 と 最後の『結果:』 の行を貼る
import json, subprocess
def run(ntr, log_every, nval=100000, ntest=200000, nlong=40000, name='run'):
    cfg = dict(L=4, n=256, ntr=ntr, nval=nval, ntest=ntest, nlong=nlong, log_every=log_every, feats=[1, 2, 3])
    p = subprocess.Popen(['python3', 'exp_ctx.py', json.dumps(cfg)], cwd='/content/ai-lab/hdc', stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    with open('/content/%s.log' % name, 'w') as f:
        for line in p.stdout:
            f.write(line)
            if any(k in line for k in ('歩 ', '結果:', '長い文脈', 'Traceback', 'Error', 'パラメータ')): print(line.rstrip()[:300], flush=True)
    p.wait()
run(3000000, 200, nval=20000, ntest=20000, nlong=5000, name='speed')

In [ ]:
# ⑤ 本番: 語数 4800万 (私が『回していい』 と言ってから)
# run(48000000, 1000, name='ld48')